In [1]:
from xmlrpc.server import SimpleXMLRPCServer
from xmlrpc.server import SimpleXMLRPCRequestHandler
import xmlrpc.client
import time
import threading
import socketserver

In [2]:
PORT = 8028

In [3]:
class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)

class ThreadedXMLRPCServer(socketserver.ThreadingMixIn, SimpleXMLRPCServer):
    """Позволяет серверу обрабатывать запросы в отдельных потоках."""
    daemon_threads = True

proxy_server = ThreadedXMLRPCServer(("localhost", PORT),
                                    requestHandler=RequestHandler,
                                    logRequests=True)

In [4]:
MAIN_SERVER_URL = "http://127.0.0.1:8008/RPC2"
STATS_SERVER_URL = "http://127.0.0.1:8038/RPC2"

thread_local = threading.local()

def get_main_server():
    """Возвращает потоко-локальное подключение к основному серверу."""
    if not hasattr(thread_local, 'main_server'):
        thread_local.main_server = xmlrpc.client.ServerProxy(MAIN_SERVER_URL)
    return thread_local.main_server

def get_stats_server():
    """Возвращает потоко-локальное подключение к серверу статистики 2."""
    if not hasattr(thread_local, 'stats_server'):
        thread_local.stats_server = xmlrpc.client.ServerProxy(STATS_SERVER_URL)
    return thread_local.stats_server

#### RATE LIMITING

**RATE LIMITING** - ограничивает количество запросов до N запросов в T секунд (например, 5 запросов в 5 секунд).  Если лимит исчерпан, то в лог должно приходить время события, тип события *too_many_requests*. По истечении ограничения по времени исходный запрос должен обработаться обычным образом с регистрацией событий

In [5]:
RATE_LIMIT_N = 5      # Максимум N запросов
RATE_LIMIT_T = 5      # В течение T секунд

In [6]:
# Список временных меток последних запросов
request_timestamps = []
# Блокировка для безопасной работы в многопоточной среде
timestamps_lock = threading.Lock()

In [7]:
def apply_rate_limit():
    """
    Применение механизма Rate Limiting.
    
    Если лимит исчерпан — ждёт, пока освободится место.
    Возвращает True, если запрос был задержан (лимит был исчерпан).
    """
    global request_timestamps
    
    was_delayed = False
    
    with timestamps_lock:
        now = time.time()
        
        # Удаление метки старше T секунд
        request_timestamps = [t for t in request_timestamps if now - t < RATE_LIMIT_T]
        
        # Если лимит исчерпан — ожидание
        if len(request_timestamps) >= RATE_LIMIT_N:
            was_delayed = True
            # Поиск самой старой метки и ожидание, пока она устареет
            oldest = min(request_timestamps)
            wait_time = RATE_LIMIT_T - (now - oldest)
            
            if wait_time > 0:
                # Логирование события too_many_requests (до ожидания)
                log_to_stats('too_many_requests', 0.0)
                print(f"[Rate Limit] Лимит исчерпан. Ожидание {wait_time:.2f} сек...")
                time.sleep(wait_time)
            
            # После ожидания снова очистка старых меткок
            now = time.time()
            request_timestamps = [t for t in request_timestamps if now - t < RATE_LIMIT_T]
        
        # Добавление текущей метки
        request_timestamps.append(time.time())
    
    return was_delayed

In [8]:
def log_to_stats(event_type, duration_ms):
    """
    Отправка события на сервер статистики 2.
    Использует потоко-локальное подключение. Если сервер недоступен — логирует в консоль и продолжает работу.
    """
    try:
        stats_srv = get_stats_server()
        stats_srv.add_log(event_type, duration_ms)
    except Exception as e:
        print(f"[WARNING] Stats2 server unavailable: {e}")

In [9]:
def create_proxy_function(func_name):
    """
    Создание функции-обёртки для проксирования вызова на основной сервер.
    """
    def proxy_func(*args, **kwargs):
        # Rate Limiting
        apply_rate_limit()
        
        # Время начала
        start_time = time.time()
        
        try:
            # Получаем потоко-локальное подключение
            main_srv = get_main_server()
            
            # Вызов функции на основном сервере
            func = getattr(main_srv, func_name)
            result = func(*args, **kwargs)
            
            # Вычисление длительности
            duration_ms = (time.time() - start_time) * 1000
            
            # Логирование успешного выполнения
            log_to_stats(func_name, duration_ms)
            
            return result
            
        except Exception as e:
            # Вычисление длительности даже при ошибке
            duration_ms = (time.time() - start_time) * 1000
            
            # Логирование ошибки
            log_to_stats(f"{func_name}:error", duration_ms)
            
            # Исключение клиенту
            raise
    
    # Имя функции для регистрации в XML-RPC
    proxy_func.__name__ = func_name
    return proxy_func

Регистация функций на сервере статистики

In [10]:
PROXY_FUNCTIONS = [
    'ping',
    'now',
    'type',              
    'sum',               
    'pow',              
    'black_list_check',
    'color_inversion',
    'binarization',
    'vertical_flip',
    'binarization_cloud',
]

In [11]:
for func_name in PROXY_FUNCTIONS:
    proxy_func = create_proxy_function(func_name)
    proxy_server.register_function(proxy_func, func_name)
    print(f"Зарегистрирована функция: {func_name}")

Зарегистрирована функция: ping
Зарегистрирована функция: now
Зарегистрирована функция: type
Зарегистрирована функция: sum
Зарегистрирована функция: pow
Зарегистрирована функция: black_list_check
Зарегистрирована функция: color_inversion
Зарегистрирована функция: binarization
Зарегистрирована функция: vertical_flip
Зарегистрирована функция: binarization_cloud


#### Запуск сервера

In [12]:
print(f"\nListening on port {PORT}...")
print(f"Прокси-сервер готов к работе")
print(f"Основной сервер: {MAIN_SERVER_URL}")
print(f"Сервер статистики 2: {STATS_SERVER_URL}")
print(f"Rate Limiting: {RATE_LIMIT_N} запросов в {RATE_LIMIT_T} секунд")

try:
    proxy_server.serve_forever()
except KeyboardInterrupt:
    print("\nОстановка прокси-сервера...")
    proxy_server.server_close()


Listening on port 8028...
Прокси-сервер готов к работе
Основной сервер: http://127.0.0.1:8008/RPC2
Сервер статистики 2: http://127.0.0.1:8038/RPC2
Rate Limiting: 5 запросов в 5 секунд


127.0.0.1 - - [18/Oct/2026 21:42:50] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 21:42:50] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 21:42:50] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 21:42:50] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 21:42:50] "POST /RPC2 HTTP/1.1" 200 -


[Rate Limit] Лимит исчерпан. Ожидание 4.99 сек...


127.0.0.1 - - [18/Oct/2026 21:42:55] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 21:42:55] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 21:42:55] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 21:42:55] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 21:42:55] "POST /RPC2 HTTP/1.1" 200 -



Остановка прокси-сервера...
